# SituationCatch-Bench — 인간 IAA (Google Colab 실행용)

이 노트북은 **인간 inter-annotator agreement(IAA)** 측정을 Colab에서 처음부터 끝까지 실행합니다.

흐름: **A) 빈 패킷·안내서 내려받아 주석자 3명에게 배부 → B) 채운 3개 파일 업로드 → C) 검증 → D) Fleiss κ·불일치표 계산 → E) 결과 내려받아 전달**

셀을 위에서부터 순서대로 `Shift+Enter`로 실행하세요. (pip 설치 불필요 — 표준 라이브러리만 사용)

## STEP 0 — 저장소 가져오기 (실행)

In [ ]:
# 리포 클론 (annotation_cli.py, 패킷, 안내서 포함)
import os
if not os.path.isdir('/content/sage'):
    !git clone --depth 1 https://github.com/leemgs/sage.git /content/sage
%cd /content/sage
!ls paper/annotation_packets

## STEP A — 빈 패킷 + 안내서 내려받기 (주석자에게 배부)
실행하면 `annotation_packets_blank.zip` 이 다운로드됩니다. 압축을 풀어 주석자 3명에게
`annotator_1.csv / annotator_2.csv / annotator_3.csv` 를 각각 하나씩, `HOW_TO_ANNOTATE.md` 와
`EXAMPLE_WALKTHROUGH.md` 를 함께 보내세요.

In [ ]:
import shutil
shutil.make_archive('/content/annotation_packets_blank','zip','paper/annotation_packets')
try:
    from google.colab import files
    files.download('/content/annotation_packets_blank.zip')
except Exception as e:
    print('수동 다운로드: 좌측 파일 탭에서 /content/annotation_packets_blank.zip', e)

### (참고) 코드북과 작성 사례를 여기서 바로 읽기

In [ ]:
print(open('paper/annotation_packets/HOW_TO_ANNOTATE.md',encoding='utf-8').read())

In [ ]:
print(open('paper/annotation_packets/EXAMPLE_WALKTHROUGH.md',encoding='utf-8').read())

## STEP B — 채운 파일 3개 업로드
주석자들이 채워 보낸 **`annotator_1.csv`, `annotator_2.csv`, `annotator_3.csv`** 를 선택해 업로드하세요.
(파일명은 반드시 그대로. 업로드하면 패킷 폴더의 빈 파일을 덮어씁니다.)

In [ ]:
from google.colab import files
import shutil, os
uploaded = files.upload()   # 파일 선택 창에서 채운 CSV 3개 선택
for name in uploaded:
    dst = os.path.join('paper/annotation_packets', os.path.basename(name))
    shutil.move(os.path.basename(name), dst)
    print('저장:', dst)

## STEP C — 제출 전 검증 (빈 칸/오탈자/허용값 확인)
`✅ 통과` 가 떠야 다음 단계로 넘어갑니다.

In [ ]:
import csv, glob
ALLOWED={
 'action':{'ANSWER','CLARIFY','ABSTAIN'},
 'temporal_state':{'relevant','stable'},
 'modality':{'confirmed','proposed'},
 'scope':{'global','limited'},
 'source_status':{'reliable','conflict'},
 'observer_state':{'shared','partial'},
 'world':{'actual','counterfactual'},
}
files_=sorted(glob.glob('paper/annotation_packets/annotator_*.csv'))
assert len(files_)>=3, f'3개 필요, 발견 {files_}'
ok=True
for f in files_:
    rows=list(csv.DictReader(open(f,encoding='utf-8-sig')))
    if len(rows)!=70: ok=False; print(f'[{f}] 70행 아님: {len(rows)}')
    for n,r in enumerate(rows,2):
        if not (r.get('answer') or '').strip(): ok=False; print(f'[{f}] {n}행 answer 비어있음 (item {r.get("item_id")})')
        for col,vals in ALLOWED.items():
            v=(r.get(col) or '').strip()
            norm = v.upper() if col=='action' else v.lower()
            allow = {x.upper() for x in vals} if col=='action' else vals
            if not v: ok=False; print(f'[{f}] {n}행 {col} 비어있음')
            elif norm not in allow: ok=False; print(f'[{f}] {n}행 {col}="{v}" 허용값 아님 {sorted(vals)}')
print('\n✅ 통과: 다음 단계로.' if ok else '\n⚠️ 위 항목 수정 후 STEP B부터 다시.')

## STEP D — Fleiss κ (일치도) + 불일치표 계산
저장소의 검증된 파이프라인(`code/annotation_cli.py`)을 그대로 사용합니다.
`--provenance human_annotations` 는 '사람이 독립 작성한 패킷'임을 명시하는 필수 플래그입니다.

In [ ]:
import os
os.makedirs('paper/results/human_iaa', exist_ok=True)
!python3 code/annotation_cli.py score --annotations paper/annotation_packets --out paper/results/human_iaa/agreement.json --provenance human_annotations
print('\n--- 불일치표 ---')
!python3 code/annotation_cli.py adjudicate --annotations paper/annotation_packets --out paper/results/human_iaa/adjudication.csv
print('done')

In [ ]:
import json
rep=json.load(open('paper/results/human_iaa/agreement.json'))
print('annotators:',rep['n_annotators'],'| items:',rep['n_items'])
print(f"{'slot':16s} {'fleiss_kappa':>12s} {'unanimous':>10s}")
for s,v in rep['slots'].items():
    print(f"{s:16s} {v['fleiss_kappa']:12.3f} {v['unanimous_rate']:10.2%}")

## STEP E — 결과 내려받아 전달
아래 두 파일을 저(어시스턴트)에게 주시면 논문(Methods/Results/Evidence-ladder + RESPONSE_TO_REVIEWERS M-B)에 반영합니다.
- `agreement.json` (Fleiss κ)
- `adjudication.csv` (불일치 문항)
채운 원본 `annotator_1/2/3.csv` 도 함께 주시면 좋습니다.

In [ ]:
import shutil
shutil.make_archive('/content/human_iaa_results','zip','paper/results/human_iaa')
# 채운 원본도 함께 묶기
import glob, zipfile
with zipfile.ZipFile('/content/human_iaa_results.zip','a') as z:
    for f in glob.glob('paper/annotation_packets/annotator_*.csv'):
        z.write(f, arcname=f.split('/')[-1])
try:
    from google.colab import files
    files.download('/content/human_iaa_results.zip')
except Exception as e:
    print('수동 다운로드: /content/human_iaa_results.zip', e)